# 5.2 Engenharia de Features — Cartola FC

Construção walk-forward do dataset de modelagem (`df_features`). Cada linha é um par `(atleta_id, rodada_id)`.

**Regra crítica — sem data leakage:** features da rodada `t` usam apenas dados das rodadas `1` a `t-1`.
`pontos_num` da rodada `t` é o **target** (y), nunca uma feature.

O preço na rodada `t` entra como preço **antes** do mercado (`preco_num - variacao_num` do snapshot),
equivalente ao preço conhecido no fechamento da rodada `t-1`.


## 5.2.0 Setup, dados e contrato de colunas


In [ ]:
from __future__ import annotations

from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import linregress

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "database" / "players.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/database/players.csv")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "database"

players = pd.read_csv(DATA_DIR / "players.csv")
games = pd.read_csv(DATA_DIR / "games.csv")

ID_COLUMNS = ["rodada_id", "atleta_id", "posicao_id", "clube_id", "status_id"]
players[ID_COLUMNS] = players[ID_COLUMNS].astype("int64")
games[
    [
        "rodada_id",
        "clube_casa_id",
        "clube_visitante_id",
        "placar_oficial_mandante",
        "placar_oficial_visitante",
    ]
] = games[
    [
        "rodada_id",
        "clube_casa_id",
        "clube_visitante_id",
        "placar_oficial_mandante",
        "placar_oficial_visitante",
    ]
].astype("int64")

print("players:", players.shape, "rounds:", sorted(int(v) for v in players["rodada_id"].unique()))
print("games:", games.shape, "rounds:", sorted(int(v) for v in games["rodada_id"].unique()))
print("unique athletes:", players["atleta_id"].nunique())


In [ ]:
POS_ONEHOT = {
    1: "pos_goleiro",
    2: "pos_lateral",
    3: "pos_zagueiro",
    4: "pos_meia",
    5: "pos_atacante",
    6: "pos_tecnico",
}
SCOUT_COLUMNS = ["G", "A", "FT", "FF", "FD", "FS", "FC", "CA", "DS", "DE", "SG", "GS"]
BLOCK_521_SCOUT_COLUMNS = [
    f"{scout}_avg_{window}" for scout in SCOUT_COLUMNS for window in ("3r", "5r")
]

BLOCK_521_COLUMNS = [
    "pts_avg_3r",
    "pts_avg_5r",
    "pts_avg_10r",
    "pts_std_3r",
    "pts_std_5r",
    "pts_max_5r",
    "pts_median_5r",
    "pts_trend_5r",
    "pts_acima_media_streak",
    "prop_jogou",
    *BLOCK_521_SCOUT_COLUMNS,
]
BLOCK_522_COLUMNS = [
    "is_mandante",
    "adversario_id",
    "adv_pagerank",
    "adv_elo",
    "adv_gols_sofridos_avg5r",
    "adv_gols_marcados_avg5r",
    "h2h_pts_avg",
    "fase_campeonato",
]
BLOCK_523_COLUMNS = ["preco_num", "preco_variacao_3r", "cb_avg_5r"]
BLOCK_524_COLUMNS = [
    "time_forma_5r",
    "time_gols_marcados_avg5r",
    "time_gols_sofridos_avg5r",
    "time_pagerank",
    "time_elo",
]
BLOCK_525_COLUMNS = [
    *POS_ONEHOT.values(),
    "status_id",
    "prob_jogar",
    "n_rodadas_jogadas",
]
FEATURE_COLUMNS = [
    "rodada_id",
    "atleta_id",
    "apelido",
    "posicao_id",
    "clube_id",
    "pontos_num",
    *BLOCK_521_COLUMNS,
    *BLOCK_522_COLUMNS,
    *BLOCK_523_COLUMNS,
    *BLOCK_524_COLUMNS,
    *BLOCK_525_COLUMNS,
]

print("n_columns:", len(FEATURE_COLUMNS))


## Helpers — grafo de resultados, PageRank e Elo

PageRank e Elo da rodada `t` são calculados só com `games` até `t-1`.
O Elo reportado é o rating **antes** da rodada `t` (todos os clubes começam em 1500).
Grafo vazio (rodada 1) → PageRank ausente (NaN); Elo permanece 1500.


In [ ]:
ELO_INITIAL = 1500.0
ELO_K = 32.0
PAGERANK_ALPHA = 0.85


def add_weighted_edge(graph: nx.DiGraph, source: int, target: int, weight: float) -> None:
    if graph.has_edge(source, target):
        graph[source][target]["weight"] += weight
    else:
        graph.add_edge(source, target, weight=weight)


def build_graph(games_hist: pd.DataFrame) -> nx.DiGraph:
    graph = nx.DiGraph()
    for row in games_hist.itertuples(index=False):
        home_score = row.placar_oficial_mandante
        away_score = row.placar_oficial_visitante
        home_id = int(row.clube_casa_id)
        away_id = int(row.clube_visitante_id)
        if home_score > away_score:
            add_weighted_edge(graph, away_id, home_id, float(home_score - away_score))
        elif away_score > home_score:
            add_weighted_edge(graph, home_id, away_id, float(away_score - home_score))
        else:
            add_weighted_edge(graph, home_id, away_id, 0.5)
            add_weighted_edge(graph, away_id, home_id, 0.5)
    return graph


def compute_pagerank(games_hist: pd.DataFrame) -> dict[int, float]:
    if games_hist.empty:
        return {}
    graph = build_graph(games_hist)
    if graph.number_of_nodes() == 0:
        return {}
    scores = nx.pagerank(graph, alpha=PAGERANK_ALPHA, weight="weight")
    return {int(club_id): float(score) for club_id, score in scores.items()}


def expected_score(rating_a: float, rating_b: float) -> float:
    return 1.0 / (1.0 + 10.0 ** ((rating_b - rating_a) / 400.0))


def update_elo(
    rating_a: float,
    rating_b: float,
    score_a: float,
    k: float = ELO_K,
) -> tuple[float, float]:
    exp_a = expected_score(rating_a, rating_b)
    new_rating_a = rating_a + k * (score_a - exp_a)
    new_rating_b = rating_b + k * ((1.0 - score_a) - (1.0 - exp_a))
    return new_rating_a, new_rating_b


def compute_elo_ratings(
    games_hist: pd.DataFrame,
    club_ids: np.ndarray,
    initial: float = ELO_INITIAL,
    k: float = ELO_K,
) -> dict[int, float]:
    ratings = {int(club_id): float(initial) for club_id in club_ids if pd.notna(club_id)}
    if games_hist.empty:
        return ratings
    ordered = games_hist.sort_values("rodada_id", kind="mergesort")
    for row in ordered.itertuples(index=False):
        home_id = int(row.clube_casa_id)
        away_id = int(row.clube_visitante_id)
        home_score = row.placar_oficial_mandante
        away_score = row.placar_oficial_visitante
        if home_score > away_score:
            score_home = 1.0
        elif away_score > home_score:
            score_home = 0.0
        else:
            score_home = 0.5
        rating_home, rating_away = update_elo(
            ratings.get(home_id, initial),
            ratings.get(away_id, initial),
            score_home,
            k,
        )
        ratings[home_id] = rating_home
        ratings[away_id] = rating_away
    return ratings


## Helpers — partidas, forma do time e confrontos

Cada jogo vira duas linhas (mandante e visitante). Forma e gols usam as últimas 5 partidas do clube até `t-1`.
O calendário da rodada `t` entra só como mando e `adversario_id` (placar de `t` não é usado).


In [ ]:
def games_to_team_matches(games_hist: pd.DataFrame) -> pd.DataFrame:
    empty = pd.DataFrame(
        {
            "rodada_id": pd.Series(dtype="int64"),
            "clube_id": pd.Series(dtype="int64"),
            "gols_marcados": pd.Series(dtype="float64"),
            "gols_sofridos": pd.Series(dtype="float64"),
            "pontos_tabela": pd.Series(dtype="int64"),
        }
    )
    if games_hist.empty:
        return empty
    home_win = games_hist["placar_oficial_mandante"] > games_hist["placar_oficial_visitante"]
    away_win = games_hist["placar_oficial_visitante"] > games_hist["placar_oficial_mandante"]
    home = pd.DataFrame(
        {
            "rodada_id": games_hist["rodada_id"].to_numpy(),
            "clube_id": games_hist["clube_casa_id"].to_numpy(),
            "gols_marcados": games_hist["placar_oficial_mandante"].to_numpy(),
            "gols_sofridos": games_hist["placar_oficial_visitante"].to_numpy(),
            "pontos_tabela": np.where(home_win, 3, np.where(away_win, 0, 1)),
        }
    )
    away = pd.DataFrame(
        {
            "rodada_id": games_hist["rodada_id"].to_numpy(),
            "clube_id": games_hist["clube_visitante_id"].to_numpy(),
            "gols_marcados": games_hist["placar_oficial_visitante"].to_numpy(),
            "gols_sofridos": games_hist["placar_oficial_mandante"].to_numpy(),
            "pontos_tabela": np.where(away_win, 3, np.where(home_win, 0, 1)),
        }
    )
    return pd.concat([home, away], ignore_index=True)


def compute_team_recent_stats(games_hist: pd.DataFrame) -> pd.DataFrame:
    matches = games_to_team_matches(games_hist)
    if matches.empty:
        return pd.DataFrame(
            {
                "clube_id": pd.Series(dtype="int64"),
                "time_forma_5r": pd.Series(dtype="float64"),
                "time_gols_marcados_avg5r": pd.Series(dtype="float64"),
                "time_gols_sofridos_avg5r": pd.Series(dtype="float64"),
            }
        )
    last_five = (
        matches.sort_values(["clube_id", "rodada_id"], kind="mergesort")
        .groupby("clube_id", sort=False)
        .tail(5)
    )
    return last_five.groupby("clube_id", as_index=False).agg(
        time_forma_5r=("pontos_tabela", "sum"),
        time_gols_marcados_avg5r=("gols_marcados", "mean"),
        time_gols_sofridos_avg5r=("gols_sofridos", "mean"),
    )


def fixture_map(games_round: pd.DataFrame) -> pd.DataFrame:
    if games_round.empty:
        return pd.DataFrame(
            {
                "clube_id": pd.Series(dtype="int64"),
                "adversario_id": pd.Series(dtype="Int64"),
                "is_mandante": pd.Series(dtype="Int64"),
            }
        )
    home = pd.DataFrame(
        {
            "clube_id": games_round["clube_casa_id"].to_numpy(),
            "adversario_id": games_round["clube_visitante_id"].to_numpy(),
            "is_mandante": 1,
        }
    )
    away = pd.DataFrame(
        {
            "clube_id": games_round["clube_visitante_id"].to_numpy(),
            "adversario_id": games_round["clube_casa_id"].to_numpy(),
            "is_mandante": 0,
        }
    )
    return pd.concat([home, away], ignore_index=True)


def opponent_by_round(games_hist: pd.DataFrame) -> pd.DataFrame:
    if games_hist.empty:
        return pd.DataFrame(
            {
                "rodada_id": pd.Series(dtype="int64"),
                "clube_id": pd.Series(dtype="int64"),
                "adversario_id": pd.Series(dtype="Int64"),
            }
        )
    home = games_hist[["rodada_id", "clube_casa_id", "clube_visitante_id"]].rename(
        columns={"clube_casa_id": "clube_id", "clube_visitante_id": "adversario_id"}
    )
    away = games_hist[["rodada_id", "clube_visitante_id", "clube_casa_id"]].rename(
        columns={"clube_visitante_id": "clube_id", "clube_casa_id": "adversario_id"}
    )
    return pd.concat([home, away], ignore_index=True)


## Helpers — histórico do atleta (janelas de k rodadas jogadas)

"Últimas k rodadas jogadas" = k aparições mais recentes no dataset, independentemente de pontuar zero.
Janela incompleta usa o que houver (`min_periods=1`). Exceção: `pts_trend_5r` exige no mínimo 3 pontos.


In [ ]:
def points_above_mean_streak(points: np.ndarray) -> int:
    streak = 0
    for index in range(len(points) - 1, -1, -1):
        if index == 0:
            break
        historical_mean = points[:index].mean()
        if points[index] > historical_mean:
            streak += 1
        else:
            break
    return int(streak)


def points_trend_slope(rounds: np.ndarray, points: np.ndarray) -> float:
    if len(points) < 3:
        return np.nan
    return float(linregress(rounds, points).slope)


def athlete_history_row(athlete_id: int, group: pd.DataFrame, round_id: int) -> dict:
    points = group["pontos_num"]
    last_five = group.tail(5)
    n_played = int((points > 0).sum())
    cost_benefit = points / group["preco_num"].replace(0, np.nan)

    row: dict = {
        "atleta_id": int(athlete_id),
        "pts_avg_3r": float(points.tail(3).mean()),
        "pts_avg_5r": float(points.tail(5).mean()),
        "pts_avg_10r": float(points.tail(10).mean()),
        "pts_std_3r": float(points.tail(3).std(ddof=0)),
        "pts_std_5r": float(points.tail(5).std(ddof=0)),
        "pts_max_5r": float(points.tail(5).max()),
        "pts_median_5r": float(points.tail(5).median()),
        "pts_trend_5r": points_trend_slope(
            last_five["rodada_id"].to_numpy(),
            last_five["pontos_num"].to_numpy(),
        ),
        "pts_acima_media_streak": points_above_mean_streak(points.to_numpy()),
        "prop_jogou": (n_played / (round_id - 1)) if round_id > 1 else np.nan,
        "preco_variacao_3r": float(group["variacao_num"].tail(3).sum()),
        "cb_avg_5r": float(cost_benefit.tail(5).mean()),
        "n_rodadas_jogadas": n_played,
    }
    for scout in SCOUT_COLUMNS:
        scout_values = group[scout]
        row[f"{scout}_avg_3r"] = float(scout_values.tail(3).mean())
        row[f"{scout}_avg_5r"] = float(scout_values.tail(5).mean())
    return row


def compute_athlete_history_features(players_hist: pd.DataFrame, round_id: int) -> pd.DataFrame:
    history_columns = ["atleta_id", *BLOCK_521_COLUMNS, "preco_variacao_3r", "cb_avg_5r", "n_rodadas_jogadas"]
    if players_hist.empty:
        empty = {"atleta_id": pd.Series(dtype="int64")}
        for column in history_columns[1:]:
            empty[column] = pd.Series(dtype="Int64" if column in {"pts_acima_media_streak", "n_rodadas_jogadas"} else "float64")
        return pd.DataFrame(empty)
    ordered = players_hist.sort_values(["atleta_id", "rodada_id"], kind="mergesort")
    rows = [
        athlete_history_row(int(athlete_id), group, round_id)
        for athlete_id, group in ordered.groupby("atleta_id", sort=False)
    ]
    return pd.DataFrame.from_records(rows)


def compute_h2h_points(
    players_hist: pd.DataFrame,
    games_hist: pd.DataFrame,
    round_df: pd.DataFrame,
) -> pd.Series:
    empty = pd.Series(np.nan, index=round_df.index, dtype="float64")
    if players_hist.empty or games_hist.empty:
        return empty
    history = players_hist.merge(
        opponent_by_round(games_hist),
        on=["rodada_id", "clube_id"],
        how="left",
    )
    h2h = (
        history.dropna(subset=["adversario_id"])
        .groupby(["atleta_id", "adversario_id"], as_index=False)["pontos_num"]
        .mean()
        .rename(columns={"pontos_num": "h2h_pts_avg"})
    )
    merged = round_df[["atleta_id", "adversario_id"]].merge(
        h2h,
        on=["atleta_id", "adversario_id"],
        how="left",
    )
    return merged["h2h_pts_avg"].to_numpy()


## 5.2.1 Desempenho histórico do jogador

Médias móveis, tendência, streak acima da média histórica (média **estrita** das aparições anteriores)
e `prop_jogou` = (rodadas com `pontos_num > 0` até `t-1`) / (`t-1`).


In [ ]:
def add_player_performance_features(
    round_df: pd.DataFrame,
    athlete_hist: pd.DataFrame,
    round_id: int,
) -> pd.DataFrame:
    out = round_df.copy()
    if athlete_hist.empty:
        for column in BLOCK_521_COLUMNS:
            out[column] = np.nan
        return out

    out = out.merge(athlete_hist[["atleta_id", *BLOCK_521_COLUMNS]], on="atleta_id", how="left")
    if round_id > 1:
        out["prop_jogou"] = out["prop_jogou"].fillna(0.0)
    return out


## 5.2.2 Contexto da partida

Mando e adversário vêm da tabela de jogos da rodada `t` (conhecidos antes do apito).
Força do adversário (PageRank, Elo, gols) e head-to-head usam apenas o histórico até `t-1`.
`fase_campeonato`: 0 nas rodadas 1–6, 1 nas rodadas 7–18.

No primeiro turno de um Brasileirão de 20 clubes quase não há reencontros, então `h2h_pts_avg` fica NaN na maior parte das linhas (exceto atletas que trocaram de clube e reencontraram um adversário já enfrentado).


In [ ]:
def add_match_context_features(
    round_df: pd.DataFrame,
    players_hist: pd.DataFrame,
    games_hist: pd.DataFrame,
    games_round: pd.DataFrame,
    pagerank: dict[int, float],
    elo: dict[int, float],
    team_stats: pd.DataFrame,
    round_id: int,
) -> pd.DataFrame:
    out = round_df.merge(fixture_map(games_round), on="clube_id", how="left")
    opponent_stats = team_stats.rename(
        columns={
            "clube_id": "adversario_id",
            "time_gols_sofridos_avg5r": "adv_gols_sofridos_avg5r",
            "time_gols_marcados_avg5r": "adv_gols_marcados_avg5r",
        }
    )[["adversario_id", "adv_gols_sofridos_avg5r", "adv_gols_marcados_avg5r"]]
    out = out.merge(opponent_stats, on="adversario_id", how="left")
    out["adv_pagerank"] = out["adversario_id"].map(pagerank)
    out["adv_elo"] = out["adversario_id"].map(elo)
    out["h2h_pts_avg"] = compute_h2h_points(players_hist, games_hist, out)
    out["fase_campeonato"] = 0 if round_id <= 6 else 1
    out["is_mandante"] = out["is_mandante"].astype("Int64")
    out["adversario_id"] = out["adversario_id"].astype("Int64")
    return out


## 5.2.3 Preço e mercado

`preco_num` aqui é o preço **pré-rodada** (`snapshot.preco_num - snapshot.variacao_num`).
`preco_variacao_3r` soma `variacao_num` das 3 aparições anteriores (não inclui a rodada `t`).
`cb_avg_5r` é a média histórica de `pontos_num / preco_num` nas últimas 5 aparições.


In [ ]:
def add_price_features(round_df: pd.DataFrame, athlete_hist: pd.DataFrame) -> pd.DataFrame:
    out = round_df.copy()
    out["preco_num"] = out["preco_num"] - out["variacao_num"]
    price_cols = ["atleta_id", "preco_variacao_3r", "cb_avg_5r"]
    if athlete_hist.empty:
        out["preco_variacao_3r"] = np.nan
        out["cb_avg_5r"] = np.nan
    else:
        out = out.merge(athlete_hist[price_cols], on="atleta_id", how="left")
    return out.drop(columns=["variacao_num"])


## 5.2.4 Features do time do atleta

Mesmo grafo e o mesmo Elo do bloco 5.2.2, agora no `clube_id` do atleta.
Forma = soma de pontos na tabela nas últimas 5 partidas (vitória=3, empate=1, derrota=0).


In [ ]:
def add_team_features(
    round_df: pd.DataFrame,
    team_stats: pd.DataFrame,
    pagerank: dict[int, float],
    elo: dict[int, float],
) -> pd.DataFrame:
    out = round_df.merge(team_stats, on="clube_id", how="left")
    out["time_pagerank"] = out["clube_id"].map(pagerank)
    out["time_elo"] = out["clube_id"].map(elo)
    return out


## 5.2.5 Posição e status

One-hots de `posicao_id` (a coluna ordinal permanece só nos identificadores).
`status_id` da rodada `t` é conhecido antes do jogo.
`prob_jogar` é a taxa empírica de `pontos_num > 0` por `status_id` no histórico até `t-1`.


In [ ]:
def add_position_status_features(
    round_df: pd.DataFrame,
    players_hist: pd.DataFrame,
    athlete_hist: pd.DataFrame,
) -> pd.DataFrame:
    out = round_df.copy()
    for pos_id, column in POS_ONEHOT.items():
        out[column] = (out["posicao_id"] == pos_id).astype(int)

    if players_hist.empty:
        out["prob_jogar"] = np.nan
    else:
        prob_map = (
            players_hist.groupby("status_id")["pontos_num"]
            .agg(lambda series: (series > 0).mean())
            .to_dict()
        )
        out["prob_jogar"] = out["status_id"].map(prob_map)

    if athlete_hist.empty:
        out["n_rodadas_jogadas"] = 0
    else:
        out = out.merge(athlete_hist[["atleta_id", "n_rodadas_jogadas"]], on="atleta_id", how="left")
        out["n_rodadas_jogadas"] = out["n_rodadas_jogadas"].fillna(0)
    out["n_rodadas_jogadas"] = out["n_rodadas_jogadas"].astype(int)
    out["pts_acima_media_streak"] = out["pts_acima_media_streak"].astype("Int64")
    return out


## Walk-forward

Para cada rodada `t`, o histórico é `rodada_id < t`. PageRank, Elo e estatísticas de clube
são calculados **uma vez** por rodada e depois unidos ao elenco.


In [ ]:
def build_round_frame(players_round: pd.DataFrame) -> pd.DataFrame:
    return players_round[
        [
            "rodada_id",
            "atleta_id",
            "apelido",
            "posicao_id",
            "clube_id",
            "pontos_num",
            "preco_num",
            "variacao_num",
            "status_id",
        ]
    ].copy()


def build_features_for_round(
    players_round: pd.DataFrame,
    players_hist: pd.DataFrame,
    games_round: pd.DataFrame,
    games_hist: pd.DataFrame,
    club_ids: np.ndarray,
    round_id: int,
) -> pd.DataFrame:
    pagerank = compute_pagerank(games_hist)
    elo = compute_elo_ratings(games_hist, club_ids)
    team_stats = compute_team_recent_stats(games_hist)
    athlete_hist = compute_athlete_history_features(players_hist, round_id)

    round_df = build_round_frame(players_round)
    round_df = add_player_performance_features(round_df, athlete_hist, round_id)
    round_df = add_match_context_features(
        round_df,
        players_hist,
        games_hist,
        games_round,
        pagerank,
        elo,
        team_stats,
        round_id,
    )
    round_df = add_price_features(round_df, athlete_hist)
    round_df = add_team_features(round_df, team_stats, pagerank, elo)
    round_df = add_position_status_features(round_df, players_hist, athlete_hist)
    return round_df.loc[:, FEATURE_COLUMNS]


club_ids = pd.unique(
    pd.concat(
        [games["clube_casa_id"], games["clube_visitante_id"], players["clube_id"]],
        ignore_index=True,
    )
)

round_frames = []
for round_id in sorted(players["rodada_id"].unique()):
    players_hist = players.loc[players["rodada_id"] < round_id]
    games_hist = games.loc[games["rodada_id"] < round_id]
    round_frames.append(
        build_features_for_round(
            players.loc[players["rodada_id"] == round_id],
            players_hist,
            games.loc[games["rodada_id"] == round_id],
            games_hist,
            club_ids,
            int(round_id),
        )
    )
    print(f"round {int(round_id):02d} done | hist_players={len(players_hist)} hist_games={len(games_hist)}")


## 5.3 Montagem, validação e persistência

Uma linha por `(atleta_id, rodada_id)`, colunas na ordem do contrato, CSV sem índice.


In [ ]:
df_features = pd.concat(round_frames, ignore_index=True)
df_features = df_features.sort_values(["rodada_id", "atleta_id"], ignore_index=True).loc[:, FEATURE_COLUMNS]

assert list(df_features.columns) == FEATURE_COLUMNS, "Unexpected column order"
assert not df_features.duplicated(["atleta_id", "rodada_id"]).any(), "Duplicate athlete-round keys"
assert len(df_features) == len(players), (
    f"Row count mismatch: features={len(df_features)} players={len(players)}"
)

output_path = DATA_DIR / "df_features.csv"
df_features.to_csv(output_path, index=False)
print("saved:", output_path)
print("df_features.shape:", df_features.shape)
print()
print("isnull().sum():")
print(df_features.isnull().sum().to_string())
print()
df_features.head(10)


In [ ]:
# Sanity check (no leakage on rolling points): at t=2, pts_avg_3r must equal pontos_num at t=1.
round_2 = df_features.loc[df_features["rodada_id"] == 2, ["atleta_id", "pts_avg_3r"]]
round_1_points = players.loc[players["rodada_id"] == 1, ["atleta_id", "pontos_num"]]
compared = round_2.merge(round_1_points, on="atleta_id", how="inner")
max_abs_diff = (compared["pts_avg_3r"] - compared["pontos_num"]).abs().max()
print("max |pts_avg_3r(t=2) - pontos_num(t=1)| among returning athletes:", max_abs_diff)
assert max_abs_diff < 1e-9

# Pre-round price must not equal post-round snapshot when variation is non-zero.
sample = df_features.merge(
    players[["rodada_id", "atleta_id", "preco_num", "variacao_num"]].rename(
        columns={"preco_num": "preco_snapshot"}
    ),
    on=["rodada_id", "atleta_id"],
    how="left",
)
reconstructed = sample["preco_snapshot"] - sample["variacao_num"]
price_diff = (sample["preco_num"] - reconstructed).abs().max()
print("max |feature preco_num - (snapshot - variacao)|:", price_diff)
assert price_diff < 1e-9
print("sanity checks passed")
